In [2]:
# ============================================================
# In-place Cleaning + Standardisation + Imputation
# Input  (your extracted CSV): data/csv/raw_data.csv   (or any csv path)
# Output (same 8 columns):     data/csv/clean_inplace.csv
# ============================================================

import os, re, difflib
import numpy as np
import pandas as pd

# ---------- PATHS (matches your folder structure) ----------
BASE_DIR = "./data"
CSV_DIR  = os.path.join(BASE_DIR, "csv")
os.makedirs(CSV_DIR, exist_ok=True)

INPUT_CSV  = os.path.join(CSV_DIR, "raw_data.csv")      # <- change if your file name differs
OUTPUT_CSV = os.path.join(CSV_DIR, "clean_inplace.csv") # <- output (same 8 columns)

# ---------- Expected schema (no extra columns in output) ----------
EXPECTED_COLS = ["report_year","report_week","district","disease","category","cases","source_pdf","uid"]

# ---------- Canonical Maharashtra districts (latest names) ----------
DISTRICTS_CANON = [
    "Ahilyanagar","Akola","Amravati","Beed","Bhandara","Buldhana","Chandrapur",
    "Chhatrapati Sambhajinagar","Dharashiv","Dhule","Gadchiroli","Gondia","Hingoli",
    "Jalgaon","Jalna","Kolhapur","Latur","Mumbai City","Mumbai Suburban","Nagpur",
    "Nanded","Nandurbar","Nashik","Palghar","Parbhani","Pune","Raigad","Ratnagiri",
    "Sangli","Satara","Sindhudurg","Solapur","Thane","Wardha","Washim","Yavatmal"
]
CANON_LOWER = {d.lower(): d for d in DISTRICTS_CANON}

# legacy / typos -> latest
DISTRICT_ALIAS = {
    "ahmednagar":"Ahilyanagar",
    "aurangabad":"Chhatrapati Sambhajinagar",
    "osmanabad":"Dharashiv",
    "palaghar":"Palghar",
    "sholapur":"Solapur",
    "bid":"Beed",
    "gadhchiroli":"Gadchiroli",
    "ghadchiroli":"Gadchiroli",
    "mumbai":"Mumbai City",
    "mumbai city":"Mumbai City",
    "mumbai suburban":"Mumbai Suburban",
}

# ---------- Helpers ----------
def clean_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def normalize_key(s: str) -> str:
    s = clean_text(s).lower()
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def best_district_match(s: str, cutoff=0.90) -> str:
    if not s:
        return ""
    s_norm = normalize_key(s)
    if not s_norm:
        return ""
    if s_norm in CANON_LOWER:
        return CANON_LOWER[s_norm]
    if s_norm in DISTRICT_ALIAS:
        return DISTRICT_ALIAS[s_norm]
    if s_norm in {"all districts","all district"}:
        return "All districts"
    m = difflib.get_close_matches(s_norm, list(CANON_LOWER.keys()), n=1, cutoff=cutoff)
    return CANON_LOWER[m[0]] if m else clean_text(s)

def disease_canonical(raw: str) -> str:
    t = clean_text(raw)
    if not t:
        return ""
    tl = t.lower()
    # remove leading roman numeral enumeration
    tl = re.sub(r"^[ivxlcdm]{1,8}[\.\)\-]?\s+", "", tl, flags=re.I)
    tl = re.sub(r"\?$", "", tl).strip()

    # vector-borne
    if re.search(r"\bdengue\b", tl): return "Dengue"
    if re.search(r"\bchikungunya\b", tl): return "Chikungunya"
    if re.search(r"\bmalaria\b|\bp\s*falciparum\b|\bp\s*vivax\b", tl): return "Malaria"
    if re.search(r"\bjapanese encephalitis\b|\bje\b", tl): return "Japanese encephalitis"
    if re.search(r"\bscrub typhus\b", tl): return "Scrub typhus"
    if re.search(r"\bkyasanur forest disease\b|\bkfd\b", tl): return "Kyasanur forest disease"
    if re.search(r"\bzika\b", tl): return "Zika"
    if re.search(r"\bwest nile\b", tl): return "West Nile fever"
    if re.search(r"\bcchf\b|crimean congo", tl): return "CCHF"
    if re.search(r"\bfilaria\b|\bfilariasis\b|\blymphatic filariasis\b", tl): return "Lymphatic filariasis"

    # water-borne
    if re.search(r"acute diarrh|\baad\b|\badd\b", tl): return "Acute diarrhoeal disease"
    if re.search(r"acute gastroenteritis|\bage\b|\bgastro-?enteritis\b", tl): return "Acute gastroenteritis"
    if re.search(r"acute watery diarr", tl): return "Acute watery diarrhoea"
    if re.search(r"\bdysentery\b|bloody diarr", tl): return "Dysentery"
    if re.search(r"\bcholera\b", tl): return "Cholera"
    if re.search(r"\bhepatitis a\b", tl): return "Hepatitis A"
    if re.search(r"\bhepatitis e\b", tl): return "Hepatitis E"
    if re.search(r"acute jaundice syndrome|\bajs\b", tl): return "Acute jaundice syndrome"
    if re.search(r"enteric fever|\btyphoid\b|\bparatyphoid\b", tl): return "Enteric fever"
    if re.search(r"\bviral hepatitis\b", tl): return "Viral hepatitis"

    # air-borne
    if re.search(r"\bmeasles rubella\b|\bmr\b", tl): return "Measles-Rubella"
    if re.search(r"\bmeasles\b", tl): return "Measles"
    if re.search(r"\brubella\b", tl): return "Rubella"
    if re.search(r"\bchickenpox\b|\bvaricella\b", tl): return "Chickenpox"
    if re.search(r"\bmumps\b", tl): return "Mumps"
    if re.search(r"\binfluenza\b|\bh1n1\b|\bh3n2\b|\bili\b|\bsari\b", tl): return "Influenza/ILI"
    if re.search(r"\bdiphtheria\b", tl): return "Diphtheria"
    if re.search(r"\bpertussis\b|whooping cough", tl): return "Pertussis"
    if re.search(r"hand foot and mouth|\bhfmd\b", tl): return "HFMD"

    return clean_text(t)

def district_needs_fix(d: str) -> bool:
    s = clean_text(d).lower()
    if s in {"", "nan", "none"}:
        return True
    if re.fullmatch(r"\d+", s):
        return True
    if s == "all districts":
        return False
    # disease leaked into district cell
    if any(k in s for k in ["dengue","malaria","chikungunya","influenza","measles","cholera","hepatitis"]):
        return True
    return s not in CANON_LOWER

def is_missing_disease(x) -> bool:
    s = clean_text(x)
    if not s or s.lower() in {"nan","none"}:
        return True
    if re.fullmatch(r"\d+", s):
        return True
    return False

def mode_series(s: pd.Series) -> str:
    s = s.dropna().astype(str).map(clean_text)
    s = s[s != ""]
    return "" if s.empty else s.value_counts().index[0]

# ---------- Main ----------
df = pd.read_csv(INPUT_CSV)

# validate
missing = [c for c in EXPECTED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Your CSV must contain: {EXPECTED_COLS}")

# 1) Standardize district + disease in-place
df["district"] = df["district"].apply(best_district_match)
df["disease"]  = df["disease"].apply(disease_canonical)

# 2) Repair shifted/missing district using UID code -> district mapping
df["uid"] = df["uid"].astype("string")
uid_code = df["uid"].str.split("/").str[1]

valid_mask = df["district"].astype(str).str.lower().isin(CANON_LOWER.keys())
tmp = df[uid_code.notna() & valid_mask].copy()
code_map = (tmp.assign(uid_code=uid_code[tmp.index])
              .groupby("uid_code")["district"]
              .agg(lambda s: s.value_counts().index[0])
              .to_dict())

new_districts = []
for d, code in zip(df["district"].tolist(), uid_code.tolist()):
    if district_needs_fix(d) and isinstance(code, str) and code in code_map:
        new_districts.append(code_map[code])
    else:
        new_districts.append(d)
df["district"] = new_districts

# 3) Fill missing disease using context (no extra columns)
mode_pdf_dist = df.groupby(["source_pdf","district"])["disease"].apply(mode_series)
mode_yw_dist  = df.groupby(["report_year","report_week","district"])["disease"].apply(mode_series)
mode_dist_cat = df.groupby(["district","category"])["disease"].apply(mode_series)
mode_cat      = df.groupby(["category"])["disease"].apply(mode_series)

disease_list = df["disease"].tolist()
for i in range(len(df)):
    if is_missing_disease(disease_list[i]):
        key1 = (df.at[i,"source_pdf"], df.at[i,"district"])
        key2 = (df.at[i,"report_year"], df.at[i,"report_week"], df.at[i,"district"])
        key3 = (df.at[i,"district"], df.at[i,"category"])
        fill = mode_pdf_dist.get(key1, "")
        if not fill:
            fill = mode_yw_dist.get(key2, "")
        if not fill:
            fill = mode_dist_cat.get(key3, "")
        if not fill:
            fill = mode_cat.get((df.at[i,"category"],), "")
        if fill:
            disease_list[i] = fill
df["disease"] = disease_list

# 4) Impute cases (hierarchical median)
df["cases"] = pd.to_numeric(df["cases"], errors="coerce")

med_dd  = df.groupby(["district","disease"])["cases"].median()
med_dc  = df.groupby(["district","category"])["cases"].median()
med_c   = df.groupby(["category"])["cases"].median()
med_all = df["cases"].median()

def impute_cases(row):
    if pd.notna(row["cases"]):
        return row["cases"]
    d, dis, cat = row["district"], row["disease"], row["category"]
    v = med_dd.get((d, dis), np.nan)
    if pd.notna(v): return v
    v = med_dc.get((d, cat), np.nan)
    if pd.notna(v): return v
    v = med_c.get((cat,), np.nan)
    if pd.notna(v): return v
    return med_all

df["cases"] = df.apply(impute_cases, axis=1)
df["cases"] = pd.to_numeric(df["cases"], errors="coerce").round().astype("Int64")

# 5) Save ONLY original columns (updated in-place)
df_out = df[EXPECTED_COLS].copy()
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print("✅ Saved:", OUTPUT_CSV)
print("Rows:", len(df_out))
print("Missing district:", int(df_out["district"].astype(str).str.strip().isin(["", "nan", "None"]).sum()))
print("Missing disease:", int(df_out["disease"].astype(str).str.strip().isin(["", "nan", "None"]).sum()))
print("Missing cases:", int(df_out["cases"].isna().sum()))

✅ Saved: ./data/csv/clean_inplace.csv
Rows: 1004
Missing district: 17
Missing disease: 0
Missing cases: 0


In [3]:
# ============================================================
# Final Model-Training Cleanup (same 8 columns, no extra flags)
# Input : data/csv/clean_inplace.csv
# Output: data/csv/clean_inplace_v2.csv
#  - fixes a few district typos/truncations
#  - drops corrupted disease text rows
#  - drops rows with missing district (unfixable early years)
#  - removes remaining duplicates
# ============================================================

import os
import pandas as pd

BASE_DIR = "./data"
CSV_DIR  = os.path.join(BASE_DIR, "csv")
os.makedirs(CSV_DIR, exist_ok=True)

INPUT_CSV  = os.path.join(CSV_DIR, "clean_inplace.csv")
OUTPUT_CSV = os.path.join(CSV_DIR, "processed.csv")

EXPECTED_COLS = ["report_year","report_week","district","disease","category","cases","source_pdf","uid"]

df = pd.read_csv(INPUT_CSV)

# --- Validate schema ---
missing = [c for c in EXPECTED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Expected: {EXPECTED_COLS}")

# --- Fix a few known district parsing artifacts ---
# (keep your standardized official names)
district_fix_map = {
    "Ahmedna gar": "Ahilyanagar",
    "Ahmednaga r": "Ahilyanagar",
    "Chhatrapati": "Chhatrapati Sambhajinagar",  # truncated
}
df["district"] = df["district"].replace(district_fix_map)

# --- Drop corrupted disease rows (parsing artifacts / paragraphs) ---
# Tune threshold if needed
df = df[df["disease"].astype(str).str.len() < 120].copy()

# --- Drop missing/blank district rows (unreliable for district models) ---
df["district"] = df["district"].astype(str).str.strip()
df = df[~df["district"].isin(["", "nan", "None"])].copy()

# --- Ensure cases is numeric Int64 ---
df["cases"] = pd.to_numeric(df["cases"], errors="coerce").round().astype("Int64")
df = df.dropna(subset=["cases"]).copy()

# --- Remove exact duplicates on the key columns ---
key_cols = ["report_year","report_week","district","disease","category","cases","uid"]
df = df.drop_duplicates(subset=key_cols, keep="first")

# --- Keep only the original 8 columns in final output ---
df_out = df[EXPECTED_COLS].copy()
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print("✅ Saved:", OUTPUT_CSV)
print("Rows:", len(df_out))
print("Years:", int(df_out["report_year"].min()), "→", int(df_out["report_year"].max()))
print("Unique districts:", df_out["district"].nunique())
print("Unique diseases:", df_out["disease"].nunique())
print("Missing cases:", int(df_out["cases"].isna().sum()))

✅ Saved: ./data/csv/processed.csv
Rows: 986
Years: 2010 → 2025
Unique districts: 34
Unique diseases: 24
Missing cases: 0
